In [1]:
import pandas as pd
import numpy as np

employees = pd.DataFrame({
    "EmployeeID": [201, 202, 203, 204, 205, 206],
    "EmployeeName": [
        "Asha", "Ben", "Chen", "Divya", "Ethan", "Farah"
    ],
    "Department": [
        "Sales", "Sales", "Marketing",
        "Sales", "Marketing", "Marketing"
    ],
    "Experience": [5, 2, 8, 4, 6, 3],
    "MonthlyTarget": [15000, 10000, 12000, 14000, 11000, 9000]
})

sales = pd.DataFrame({
    "SaleID": range(701, 713),
    "EmployeeID": [
        201, 202, 201, 203, 204, 202,
        205, 203, 201, 204, 205, 207
    ],
    "Product": [
        "Laptop", "Phone", "Tablet", "Laptop",
        "Phone", "Laptop", "Tablet", "Phone",
        "Laptop", "Tablet", "Laptop", "Phone"
    ],
    "Revenue": [
        5000, 3000, 2500, 6000, 4500, 4000,
        3500, 2800, 5500, 3200, 4800, 2000
    ]
})

In [2]:
#Merge employees and sales using a left join.
employee_sales = pd.merge(
    employees,
    sales,
    on="EmployeeID",
    how="left"
)

employee_sales

,EmployeeID,EmployeeName,Department,Experience,MonthlyTarget,SaleID,Product,Revenue
0,201,Asha,Sales,5,15000,701.0,Laptop,5000.0
1,201,Asha,Sales,5,15000,703.0,Tablet,2500.0
2,201,Asha,Sales,5,15000,709.0,Laptop,5500.0
3,202,Ben,Sales,2,10000,702.0,Phone,3000.0
4,202,Ben,Sales,2,10000,706.0,Laptop,4000.0
5,203,Chen,Marketing,8,12000,704.0,Laptop,6000.0
6,203,Chen,Marketing,8,12000,708.0,Phone,2800.0
7,204,Divya,Sales,4,14000,705.0,Phone,4500.0
8,204,Divya,Sales,4,14000,710.0,Tablet,3200.0
9,205,Ethan,Marketing,6,11000,707.0,Tablet,3500.0


In [3]:
#Calculate total revenue generated by each employee.
revenue_by_employee = sales.groupby("EmployeeID")["Revenue"].sum()
print(revenue_by_employee)

EmployeeID
201    13000
202     7000
203     8800
204     7700
205     8300
207     2000
Name: Revenue, dtype: int64


In [4]:
#Include employees with zero sales in the summary.
employee_totals = sales.groupby("EmployeeID", as_index=False).agg(
    TotalRevenue=("Revenue", "sum"),
    TransactionCount=("SaleID", "count")
)

employee_summary = pd.merge(
    employees,
    employee_totals,
    on="EmployeeID",
    how="left"
)

employee_summary[["TotalRevenue", "TransactionCount"]] = (
    employee_summary[["TotalRevenue", "TransactionCount"]]
    .fillna(0)
    .astype(int)
)

employee_summary

,EmployeeID,EmployeeName,Department,Experience,MonthlyTarget,TotalRevenue,TransactionCount
0,201,Asha,Sales,5,15000,13000,3
1,202,Ben,Sales,2,10000,7000,2
2,203,Chen,Marketing,8,12000,8800,2
3,204,Divya,Sales,4,14000,7700,2
4,205,Ethan,Marketing,6,11000,8300,2
5,206,Farah,Marketing,3,9000,0,0


In [5]:
#Calculate total revenue by department.
department_revenue = employee_sales.groupby(
    "Department"
)["Revenue"].sum()

department_revenue

Department
Marketing    17100.0
Sales        27700.0
Name: Revenue, dtype: float64

In [6]:
#Calculate the number of sales transactions by employee.
sales.groupby("EmployeeID")["SaleID"].count()

EmployeeID
201    3
202    2
203    2
204    2
205    2
207    1
Name: SaleID, dtype: int64

In [7]:
#Find the highest-revenue employee.
employee_summary.loc[
    employee_summary["TotalRevenue"].idxmax(),
    ["EmployeeName", "TotalRevenue"]
]

EmployeeName     Asha
TotalRevenue    13000
Name: 0, dtype: object

In [8]:
#Calculate each employee's target achievement percentage:[{Achievement %} ={Total Revenue}/{{Monthly Target}}*100]
employee_summary["AchievementPct"] = (
    employee_summary["TotalRevenue"]
    / employee_summary["MonthlyTarget"]
    * 100
).round(2)

employee_summary[
    ["EmployeeName", "TotalRevenue", "MonthlyTarget", "AchievementPct"]
]

,EmployeeName,TotalRevenue,MonthlyTarget,AchievementPct
0,Asha,13000,15000,86.67
1,Ben,7000,10000,70.00
2,Chen,8800,12000,73.33
3,Divya,7700,14000,55.00
4,Ethan,8300,11000,75.45
5,Farah,0,9000,0.00


In [9]:
#Create a TargetMet column that is True when achievement is at least 100%.
employee_summary["TargetMet"] = (
    employee_summary["AchievementPct"] >= 100
)

employee_summary[["EmployeeName", "AchievementPct", "TargetMet"]]

,EmployeeName,AchievementPct,TargetMet
0,Asha,86.67,False
1,Ben,70.00,False
2,Chen,73.33,False
3,Divya,55.00,False
4,Ethan,75.45,False
5,Farah,0.00,False


In [10]:
#Calculate total revenue by department and product.
department_product = pd.merge(
    sales,
    employees[["EmployeeID", "Department"]],
    on="EmployeeID",
    how="inner"
)

department_product.groupby(
    ["Department", "Product"]
)["Revenue"].sum()

Department  Product
Marketing   Laptop     10800
            Phone       2800
            Tablet      3500
Sales       Laptop     14500
            Phone       7500
            Tablet      5700
Name: Revenue, dtype: int64

In [11]:
#Identify employees who generated more than their monthly target.
employee_summary[
    employee_summary["AchievementPct"] > 100
]

,EmployeeID,EmployeeName,Department,Experience,MonthlyTarget,TotalRevenue,TransactionCount,AchievementPct,TargetMet


In [12]:
#Find products ranked by total revenue, highest first.
sales.groupby("Product")["Revenue"].sum().sort_values(
    ascending=False
)

Product
Laptop    25300
Phone     12300
Tablet     9200
Name: Revenue, dtype: int64

In [13]:
#Create a final employee performance summary containing employee name, department, experience, total revenue, 
# transaction count, achievement percentage, and target status.

final_summary = employee_summary[
    [
        "EmployeeName",
        "Department",
        "Experience",
        "TotalRevenue",
        "TransactionCount",
        "AchievementPct",
        "TargetMet"
    ]
].sort_values("TotalRevenue", ascending=False)

final_summary = final_summary.reset_index(drop=True)

final_summary

,EmployeeName,Department,Experience,TotalRevenue,TransactionCount,AchievementPct,TargetMet
0,Asha,Sales,5,13000,3,86.67,False
1,Chen,Marketing,8,8800,2,73.33,False
2,Ethan,Marketing,6,8300,2,75.45,False
3,Divya,Sales,4,7700,2,55.00,False
4,Ben,Sales,2,7000,2,70.00,False
5,Farah,Marketing,3,0,0,0.00,False
